In [ ]:
import pandas as pd

file_path = r"C:\online_food_delivery\data\raw\Online_Food_Delivery_Analysis.csv"

df = pd.read_csv(file_path)

print("Dataset Loaded Successfully ✅")
print("Shape:", df.shape)
df.head()

Dataset Loaded Successfully ✅
Shape: (100000, 25)


,Order_ID,Customer_ID,Customer_Age,Customer_Gender,City,Area,Restaurant_ID,Restaurant_Name,Cuisine_Type,Order_Date,...,Final_Amount,Payment_Mode,Order_Status,Cancellation_Reason,Delivery_Partner_ID,Delivery_Rating,Restaurant_Rating,Order_Day,Peak_Hour,Profit_Margin
0,ORD000001,CUST6948,19.0,Male,NaN,Central,RES936,Restaurant_29,Chinese,10/20/2024,...,NaN,UPI,Delivered,NaN,DP563,5.0,4.4,Weekend,True,0.13
1,ORD000002,CUST6515,NaN,Female,Chennai,North,RES689,Restaurant_419,Chinese,8/12/2024,...,4849.0,COD,Delivered,NaN,DP369,5.0,4.7,Weekday,True,0.48
2,ORD000003,CUST1765,NaN,Male,Delhi,NaN,RES723,Restaurant_244,Arabian,12/8/2024,...,737.0,Wallet,Delivered,NaN,DP580,4.0,4.9,Weekend,True,0.08
3,ORD000004,CUST2744,NaN,Male,Mumbai,Central,RES951,Restaurant_178,Chinese,10/8/2024,...,NaN,UPI,Cancelled,Late Delivery,DP155,2.0,3.4,Weekday,NaN,0.04
4,ORD000005,CUST4389,57.0,Female,Chennai,South,RES419,Restaurant_262,Chinese,2/4/2024,...,352.0,Card,Delivered,NaN,DP728,2.0,4.4,Weekend,False,0.12


In [2]:
#2
# # 1️⃣ Basic Info
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# 2️⃣ Column Names
print("\nColumns List:")
print(df.columns.tolist())

# 3️⃣ Data Types
print("\nData Types:")
print(df.dtypes)

# 4️⃣ Missing Values
print("\nMissing Values Count:")
print(df.isnull().sum().sort_values(ascending=False))

Rows: 100000
Columns: 25

Columns List:
['Order_ID', 'Customer_ID', 'Customer_Age', 'Customer_Gender', 'City', 'Area', 'Restaurant_ID', 'Restaurant_Name', 'Cuisine_Type', 'Order_Date', 'Order_Time', 'Delivery_Time_Min', 'Distance_km', 'Order_Value', 'Discount_Applied', 'Final_Amount', 'Payment_Mode', 'Order_Status', 'Cancellation_Reason', 'Delivery_Partner_ID', 'Delivery_Rating', 'Restaurant_Rating', 'Order_Day', 'Peak_Hour', 'Profit_Margin']

Data Types:
Order_ID                object
Customer_ID             object
Customer_Age           float64
Customer_Gender         object
City                    object
Area                    object
Restaurant_ID           object
Restaurant_Name         object
Cuisine_Type            object
Order_Date              object
Order_Time              object
Delivery_Time_Min      float64
Distance_km            float64
Order_Value            float64
Discount_Applied       float64
Final_Amount           float64
Payment_Mode            object
Order_Status 

In [3]:
#3
null_counts = df.isnull().sum()
null_df = pd.DataFrame({
    "Missing_Count": null_counts,
    "Missing_%": (null_counts / len(df)) * 100
}).sort_values(by="Missing_Count", ascending=False)

null_df

,Missing_Count,Missing_%
Cancellation_Reason,90969,90.969
Final_Amount,55697,55.697
Customer_Age,50093,50.093
Distance_km,33470,33.470
Delivery_Time_Min,33359,33.359
Order_Value,33327,33.327
Peak_Hour,32962,32.962
Customer_Gender,24856,24.856
Payment_Mode,19911,19.911
Cuisine_Type,16885,16.885


In [ ]:
#4
df['Order_Status'].value_counts()

Order_Status
Delivered    84964
Cancelled    15036
Name: count, dtype: int64

In [5]:
#5
print("Order Status Distribution:")
print(df['Order_Status'].value_counts(normalize=True) * 100)

Order Status Distribution:
Order_Status
Delivered    84.964
Cancelled    15.036
Name: proportion, dtype: float64


In [6]:
#6
df[(df['Order_Status'] == 'Delivered') & (df['Final_Amount'].isnull())].shape

(47372, 25)

In [7]:

#7
df[(df['Order_Status'] == 'Cancelled') & (df['Final_Amount'].isnull())].shape

(8325, 25)

In [8]:
#8
# Check if Final_Amount matches calculation where it exists
sample_check = df[df['Final_Amount'].notnull()].copy()

sample_check['Calculated_Final'] = sample_check['Order_Value'] - sample_check['Discount_Applied']

# Compare difference
(sample_check['Final_Amount'] - sample_check['Calculated_Final']).abs().mean()

np.float64(0.0)

In [9]:
#code 9 Recalculate Final_Amount for delivered orders where it is missing

mask = (df['Order_Status'] == 'Delivered') & (df['Final_Amount'].isnull())

df.loc[mask, 'Final_Amount'] = (
    df.loc[mask, 'Order_Value'] - df.loc[mask, 'Discount_Applied']
)

# Verify again
df['Final_Amount'].isnull().sum()

np.int64(46115)

In [10]:
# code 10Check missing in components for delivered orders
df[(df['Order_Status'] == 'Delivered') & 
   ((df['Order_Value'].isnull()) | (df['Discount_Applied'].isnull()))].shape

(37790, 25)

In [11]:
df['Discount_Applied'].describe()

count    83285.000000
mean        93.936243
std        108.209904
min          0.000000
25%         20.000000
50%         50.000000
75%        100.000000
max        300.000000
Name: Discount_Applied, dtype: float64

In [12]:
# 11 Rows where Discount_Applied is missing but Final_Amount exists
check_df = df[(df['Discount_Applied'].isnull()) & (df['Final_Amount'].notnull())]

check_df[['Order_Value', 'Final_Amount']].head()

,Order_Value,Final_Amount


In [13]:
# Check difference for those rows
(check_df['Order_Value'] - check_df['Final_Amount']).unique()

array([], dtype=float64)

In [14]:
df['Discount_Applied'] = df['Discount_Applied'].fillna(0)
df['Discount_Applied'].isnull().sum()

np.int64(0)

In [15]:
#Recalculate final amount
mask = (df['Order_Status'] == 'Delivered') & (df['Final_Amount'].isnull())

df.loc[mask, 'Final_Amount'] = (
    df.loc[mask, 'Order_Value'] - df.loc[mask, 'Discount_Applied']
)

df['Final_Amount'].isnull().sum()

np.int64(36646)

In [16]:
df[(df['Final_Amount'].isnull()) & (df['Order_Value'].isnull())].shape

(33327, 25)

In [17]:
df[(df['Order_Status'] == 'Delivered') & (df['Order_Value'].isnull())].shape

(28321, 25)

In [18]:
df['Order_Value'].describe()

count    66673.000000
mean      2081.830126
std       1553.628891
min        150.000000
25%        673.000000
50%       1197.000000
75%       3494.000000
max       5000.000000
Name: Order_Value, dtype: float64

In [19]:
median_value = df['Order_Value'].median()

df['Order_Value'] = df['Order_Value'].fillna(median_value)

df['Order_Value'].isnull().sum()

np.int64(0)

In [20]:
mask = (df['Order_Status'] == 'Delivered') & (df['Final_Amount'].isnull())

df.loc[mask, 'Final_Amount'] = (
    df.loc[mask, 'Order_Value'] - df.loc[mask, 'Discount_Applied']
)

df['Final_Amount'].isnull().sum()

np.int64(8325)

In [21]:
df[(df['Order_Status'] == 'Delivered') & (df['Delivery_Time_Min'].isnull())].shape

(28329, 25)

In [22]:
df[(df['Order_Status'] == 'Delivered') & 
   (df['Delivery_Time_Min'].isnull()) & 
   (df['Distance_km'].isnull())].shape

(9503, 25)

In [23]:
df[['Distance_km', 'Delivery_Time_Min']].corr()

,Distance_km,Delivery_Time_Min
Distance_km,1.000000,0.002599
Delivery_Time_Min,0.002599,1.000000


In [24]:
df['Delivery_Time_Min'].describe()

count    66641.000000
mean       127.475923
std         90.805839
min         20.000000
25%         45.000000
50%        120.000000
75%        210.000000
max        300.000000
Name: Delivery_Time_Min, dtype: float64

In [25]:
median_delivery_time = df['Delivery_Time_Min'].median()

mask = (df['Order_Status'] == 'Delivered') & (df['Delivery_Time_Min'].isnull())

df.loc[mask, 'Delivery_Time_Min'] = median_delivery_time

df['Delivery_Time_Min'].isnull().sum()

np.int64(5030)

In [26]:
df[(df['Order_Status'] == 'Delivered') & (df['Distance_km'].isnull())].shape

(28352, 25)

In [27]:
df['Distance_km'].describe()

count    66530.000000
mean        16.449242
std         12.256742
min          1.000000
25%          5.470000
50%          9.970000
75%         27.430000
max         40.000000
Name: Distance_km, dtype: float64

In [28]:
median_distance = df['Distance_km'].median()

mask = (df['Order_Status'] == 'Delivered') & (df['Distance_km'].isnull())

df.loc[mask, 'Distance_km'] = median_distance

df['Distance_km'].isnull().sum()

np.int64(5118)

In [29]:
df['Customer_Age'].describe()

count    49907.000000
mean        38.976516
std         12.372157
min         18.000000
25%         28.000000
50%         39.000000
75%         50.000000
max         60.000000
Name: Customer_Age, dtype: float64

In [30]:
df['Customer_Age'].value_counts().head()

Customer_Age
27.0    1260
33.0    1250
49.0    1243
34.0    1208
23.0    1201
Name: count, dtype: int64

In [31]:
median_age = df['Customer_Age'].median()

df['Customer_Age'] = df['Customer_Age'].fillna(median_age)

df['Customer_Age'].isnull().sum()

np.int64(0)

In [32]:
df['Customer_Gender'].value_counts(dropna=False)

Customer_Gender
Other     25135
Female    25031
Male      24978
NaN       24856
Name: count, dtype: int64

In [33]:
df['Customer_Gender'] = df['Customer_Gender'].fillna('Unknown')

df['Customer_Gender'].value_counts()

Customer_Gender
Other      25135
Female     25031
Male       24978
Unknown    24856
Name: count, dtype: int64

In [34]:
df['City'].value_counts(dropna=False)

City
Hyderabad    16884
Bangalore    16732
NaN          16726
Delhi        16695
Mumbai       16493
Chennai      16470
Name: count, dtype: int64

In [35]:
df['City'] = df['City'].fillna('Unknown')

df['City'].value_counts()

City
Hyderabad    16884
Bangalore    16732
Unknown      16726
Delhi        16695
Mumbai       16493
Chennai      16470
Name: count, dtype: int64

In [36]:
# Check relationship between City and Area missing
df[(df['City'] == 'Unknown') & (df['Area'].isnull())].shape

(2724, 25)

In [37]:
df['Area'].value_counts(dropna=False)

Area
South      16725
NaN        16685
North      16675
West       16653
East       16643
Central    16619
Name: count, dtype: int64

In [38]:
df['Area'] = df['Area'].fillna('Unknown')

df['Area'].value_counts()

Area
South      16725
Unknown    16685
North      16675
West       16653
East       16643
Central    16619
Name: count, dtype: int64

In [39]:
df['Cuisine_Type'].value_counts(dropna=False)

Cuisine_Type
NaN        16885
Indian     16685
Arabian    16658
Chinese    16651
Mexican    16602
Italian    16519
Name: count, dtype: int64

In [40]:
df['Cuisine_Type'] = df['Cuisine_Type'].fillna('Unknown')

df['Cuisine_Type'].value_counts()

Cuisine_Type
Unknown    16885
Indian     16685
Arabian    16658
Chinese    16651
Mexican    16602
Italian    16519
Name: count, dtype: int64

In [41]:
df['Payment_Mode'].value_counts(dropna=False)

Payment_Mode
Card      20094
Wallet    20086
COD       19977
UPI       19932
NaN       19911
Name: count, dtype: int64

In [42]:
df['Payment_Mode'] = df['Payment_Mode'].fillna('Unknown')

df['Payment_Mode'].value_counts()

Payment_Mode
Card       20094
Wallet     20086
COD        19977
UPI        19932
Unknown    19911
Name: count, dtype: int64

In [43]:
df['Delivery_Rating'].describe()

df['Delivery_Rating'].value_counts(dropna=False)

Delivery_Rating
2.0    16896
4.0    16885
1.0    16818
NaN    16523
5.0    16470
3.0    16408
Name: count, dtype: int64

In [44]:
df[(df['Order_Status'] == 'Delivered') & (df['Delivery_Rating'].isnull())].shape

(14048, 25)

In [45]:
df['Delivery_Rating'] = df['Delivery_Rating'].fillna('No Rating')

df['Delivery_Rating'].value_counts()

Delivery_Rating
2.0          16896
4.0          16885
1.0          16818
No Rating    16523
5.0          16470
3.0          16408
Name: count, dtype: int64

In [46]:
df['Order_Time'].head()

0    0:00
1    0:00
2    0:00
3    0:00
4    0:00
Name: Order_Time, dtype: object

In [47]:
df['Order_Time'].value_counts()

Order_Time
0:00    98002
Name: count, dtype: int64

In [48]:
df['Peak_Hour'].value_counts(dropna=False)

Peak_Hour
False    33585
True     33453
NaN      32962
Name: count, dtype: int64

In [49]:
df['Peak_Hour'] = df['Peak_Hour'].astype('object')

df['Peak_Hour'] = df['Peak_Hour'].fillna('Unknown')

df['Peak_Hour'].value_counts()

Peak_Hour
False      33585
True       33453
Unknown    32962
Name: count, dtype: int64

In [50]:
df.isnull().sum().sort_values(ascending=False)

Cancellation_Reason    90969
Final_Amount            8325
Distance_km             5118
Delivery_Time_Min       5030
Order_Time              1998
Order_Date              1014
Customer_Age               0
Customer_ID                0
Order_ID                   0
Cuisine_Type               0
Restaurant_Name            0
Restaurant_ID              0
Area                       0
City                       0
Customer_Gender            0
Discount_Applied           0
Order_Value                0
Payment_Mode               0
Order_Status               0
Delivery_Partner_ID        0
Delivery_Rating            0
Restaurant_Rating          0
Order_Day                  0
Peak_Hour                  0
Profit_Margin              0
dtype: int64

In [51]:
df['Order_Date'].head()

0    10/20/2024
1     8/12/2024
2     12/8/2024
3     10/8/2024
4      2/4/2024
Name: Order_Date, dtype: object

In [52]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='%m/%d/%Y', errors='coerce')

df['Order_Date'].isnull().sum()

np.int64(1014)

In [53]:
df = df.dropna(subset=['Order_Date'])

df['Order_Date'].isnull().sum()

np.int64(0)

In [54]:
df.shape

(98986, 25)

In [55]:
df.to_csv("../data/processed/cleaned_online_food_delivery.csv", index=False)